이 자료는 Hugging Face Pipeline API를 이용한 챗봇 튜토리얼로 작성되었습니다.  
2025년 9월 14일에 정상 동작을 확인하였습니다.  

1. Hugging Face 회원가입(Sign Up)

    - [Hugging Face 바로가기: https://huggingface.co/](https://huggingface.co/)  
    - Sign Up 따라하기  

2. Hugging Face의 Access Token 생성  

    - 우측상단 (三) 메뉴 -> Settings -> Tokens -> +Create new token
    - 읽기 전용(Read) / 쓰기(Write) / 관리자(Admin) 중 목적에 맞게 발급  
    - 발급된 코드는 다시 볼 수 없기 때문에 별도로 보관  

In [4]:
from huggingface_hub import login
login(token="Your Hugging Face Access Token")

3. Transformer 설치  

In [5]:
!pip install --upgrade transformers

4. Git credential 설정 하기  

In [6]:
!git config --global credential.helper store

5. Hugging Face 로그인 상태 확인  

In [7]:
!hf auth whoami

user:  gislee


6. 질의응답(Q&A) 모델 불러오기 (bert-large-uncased-whole-word-masking-finetuned-squad)

In [8]:
from transformers import pipeline

qa = pipeline("question-answering")

context = """AI agents are intelligent systems that can perceive the environment,
make decisions, and take actions autonomously. They are increasingly powered by
large language models (LLMs) such as GPT or LLaMA."""

question = "What powers modern AI agents?"
result = qa(question=question, context=context)
print("질문:", question)
print("답변:", result['answer'])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


질문: What powers modern AI agents?
답변: large language models


8. Hugging Face Pipeline API를 이용한 챗봇

In [9]:
# 1) Colab.로컬 공통: 설치
!pip -q install "transformers>=4.44.0" sentencepiece gradio

In [12]:
# 2) 간단 챗봇 (Pipeline + 대화 이력 유지)
import torch
from transformers import pipeline, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = 0 if torch.cuda.is_available() else -1

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
chatbot = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=tok,
    device=device
)

SYSTEM_PROMPT = (
    "You are a helpful, concise assistant for a university AI Agent course. "
    "Answer in the user's language (Korean or English)."
)

def format_history(history, user_text, system_prompt=SYSTEM_PROMPT):
    """
    history: list[tuple[str,str]] = [(user, assistant), ...]
    user_text: 현재 입력
    """
    lines = [f"System: {system_prompt}"]
    for u, a in history:
        lines.append(f"User: {u}")
        lines.append(f"Assistant: {a}")
    lines.append(f"User: {user_text}")
    lines.append("Assistant:")
    return "\n".join(lines)

def generate_reply(history, user_text):
    prompt = format_history(history, user_text)
    out = chatbot(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=tok.eos_token_id
    )[0]["generated_text"]
    # 간단 파싱: 마지막 "Assistant:" 뒤 텍스트를 응답으로 사용
    reply = out.split("Assistant:")[-1].strip()
    # 길이 과도 시 컷
    return reply[:1200]

# 🔁 터미널 대화 루프 (원하면 실행)
history = []
print("간단 챗봇 시작! 종료: exit/quit\n")
while True:
    user = input("You: ").strip()
    if user.lower() in {"exit","quit"}:
        print("Bye!")
        break
    bot = generate_reply(history, user)
    history.append((user, bot))
    print("Bot:", bot)
    print("\n")

Device set to use cuda:0


간단 챗봇 시작! 종료: exit/quit

You: AI Agent 개념을 설명해줘
Bot: AI Agent는 Artificial Intelligence를 Agent로 해석한 것을 말합니다. 이는 인공지능의 한 형태로서, 특정 업무를 수행하기 위해 설계된 컴퓨터 프로그램입니다. 이는 인간이 직접적으로 대화하거나 행동을 제어하는 것이 아니라, 자동으로 결정을 내리고 행동을 취하도록 설계된 프로그램입니다.

AI Agent는 여러 분야에서 사용되며, 주요 사항은 다음과 같습니다:

1. **기타 업무**: 물리적 작업, 정보 수집, 데이터 분석 등.
2. **미래 예측**: 미래的情況이나 시나리오에 대한 예측을 도와줍니다.
3. **물리적 운송**: 차량이나 항공기 등의 운송을 자동화합니다.
4. **보안**: 공격과 위협을 감지하고 대응합니다.
5. **교육**: 학습 및 학습 방법을 개선합니다.

AI Agent는 컴퓨터 프로그램으로, 사람이 아닌 자동으로 결정을 내리고 행동을 취하게 합니다. 이는 인간의 인간적인 이해와 감정을 보다 쉽게 이해할 수 있도록 도와줍니다.


You: AI Agent와 LLM 과의 관계는?
Bot: AI Agent와 Large Language Model (LLM)은 서로 다른 기술 분야에서 사용되는 용어입니다. 그러나 LLM과 AI Agent 사이에는 상호 연관성이 있습니다.

1. **LLM과 AI Agent의 유사점**:
   - **Both are based on AI**: Both LLM and AI Agents rely on machine learning algorithms to process information and make decisions.
   - **Both can interact with humans**: Both can be programmed to engage in conversations or other forms of human-computer interaction.

2. **LLM과 AI Age

In [13]:
# 3) (선택) Gradio 웹 UI
import gradio as gr

history = []  # [(user, bot), ...]

def respond(msg, chat_hist):
    global history
    # gradio chat_hist: [[user, bot], ...]
    history = [(u or "", b or "") for (u, b) in (chat_hist or [])]
    bot = generate_reply(history, msg)
    history.append((msg, bot))
    chat_hist = (chat_hist or []) + [[msg, bot]]
    return "", chat_hist

def clear():
    global history
    history = []
    return []

with gr.Blocks() as demo:
    gr.Markdown("## 🤗 Hugging Face pipeline 기반 간단 챗봇")
    gr.Markdown(f"- 모델: **{MODEL_NAME}**  |  한국어/영어 자동 응답")
    chat = gr.Chatbot(height=380)
    txt = gr.Textbox(placeholder="질문을 입력하세요…", autofocus=True)
    btn = gr.Button("대화 리셋")

    txt.submit(respond, [txt, chat], [txt, chat])
    btn.click(clear, None, chat)

demo.launch(debug=False)

/tmp/ipython-input-387106240.py:23: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat = gr.Chatbot(height=380)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8abe558f90ff9a5aaa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
